# New Notebook file

In [ ]:
import random
import string

from pyspark.sql import SparkSession
from pyspark.sql.functions import abs, avg, array, explode, col

spark = (
    SparkSession.builder
    .appName("Skewed Joins")
    .master("local[*]")
    .config("spark.sql.autoBroadcastJoinThreshold", -1)
    .getOrCreate()
)

An online store selling selling gaming laptops.
2 laptops are "similar" if they have the same make & model, but proc speed within 0.1

For each laptop configuration, we are interested in the average sale price of "similar" models

Acer Predator, 2.9 Ghz, uid: asldf -> avg sale price of all Acer Predators with CPU speed between 2.8 and 3.0 Ghz

In [ ]:

LAPTOP_MODELS = [
    ("Razer", "Blade"),
    ("Alienware", "Area-51"),
    ("HP", "Omen"),
    ("Acer", "Predator"),
    ("Asus", "ROG"),
    ("Lenovo", "Legion"),
    ("MSI", "Raider"),
]


def random_laptop_model(uniform=False):
    if not uniform and random.random() < 0.5:
        return LAPTOP_MODELS[0]
    return random.choice(LAPTOP_MODELS)

def random_proc_speed():
    return float(f"3.{random.randint(0, 8)}")

def random_registration():
    return "".join(random.choices(string.ascii_letters + string.digits, k=7))

def random_price():
    return 500 + random.randint(0, 1499)

def random_laptop():
    make, model = random_laptop_model()
    return (random_registration(), make, model, random_proc_speed())

def random_laptop_offer():
    make, model = random_laptop_model()
    return (make, model, random_proc_speed(), float(random_price()))

In [ ]:

laptops = spark.createDataFrame(
    [random_laptop() for _ in range(40000)],
    schema=["registration","make", "model", "procSpeed"] 
)

laptopOffers = spark.createDataFrame(
    [random_laptop_offer() for _ in range(100000)],
    schema=["make", "model", "procSpeed", "salePrice"]
)



laptops.printSchema()
laptopOffers.printSchema()

In [ ]:

joined = (
        laptops
            .join(laptopOffers, on=["make", "model"])
            .filter(abs(laptopOffers["procSpeed"] - laptops["procSpeed"]) <= 0.1)
            .groupBy("registration")
            .agg(avg("salePrice").alias("averagePrice"))
)

joined.show()
joined.explain()

Notice that the job takes a while and its not obvious from physical plan why
Even the shuffle size is pretty small
However if you sort tasks in that stage by duration you will notice most tasks are very fast and
there is one task that takes forever. this is called a straggling task!
You can use summary metrics table to find straggling tasks max should be close to medium in stages
without straggling tasks

Why? 50% of the data is same make and model so 50% have to stay in same executor.
Normally you can only tell this by analyzing the data itself and analyzing spark ui.

In [ ]:

laptops2 = (
        laptops.withColumn("procSpeed", explode(array(
        col("procSpeed") - 0.1,
        col("procSpeed"),
        col("procSpeed") + 0.1,
    )))   
)

joined2 = (
    laptops2.join(laptopOffers, on=["make", "model", "procSpeed"])
    .groupBy("registration")
    .agg(avg("salePrice").alias("averagePrice"))
)

joined2.explain()
joined2.show()


now we are joining in 3 columns instead of 2 so this will spread that large partition of make and model
across proc speeds so this will make a lot more of a uniform pool and skew is more or less eliminated.

our task distribution should be much better might still have some skew but should be better.

The only difference in our query plan is a new generate for the explode
so spark code is less good at telling you there is skew as much as spark ui and looking at data.